# ML Pipeline: Inspection Outcome Prediction

Loads `data/feature_matrix.csv`, applies a time-based train/test split, and evaluates
two model types (LogisticRegression, RandomForestClassifier) with and without feature
selection. Goal: beat the majority-class baseline of 62.0%.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

BASE = Path('../data')

## 1. Load Data

In [2]:
df = pd.read_csv(BASE / 'feature_matrix.csv', parse_dates=['inspection_date'])

# Drop columns that are not model inputs
DROP_COLS = ['ElevatingDevicesNumber', 'InspectionNumber', 'inspection_date', 'location']
df = df.drop(columns=DROP_COLS)

# Convert bool columns to int (required by some sklearn estimators)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'Shape: {df.shape}')
print(f'Features: {df.shape[1] - 1}')
print(f'\nTarget distribution:')
print(df['outcome_binary'].value_counts())

Shape: (141789, 31)
Features: 30

Target distribution:
outcome_binary
Needs Action    87877
Passed          53912
Name: count, dtype: int64


## 2. Baseline Score

In [3]:
majority_class = df['outcome_binary'].value_counts().idxmax()
baseline = df['outcome_binary'].value_counts().max() / len(df)
print(f'Majority class: {majority_class!r}')
print(f'Baseline accuracy (always predict majority): {baseline:.3f} ({baseline:.1%})')

Majority class: 'Needs Action'
Baseline accuracy (always predict majority): 0.620 (62.0%)


## 3. Time-Based Train/Test Split

Records are sorted by `inspection_date` and split 80/20. The model trains on earlier
inspections and is tested on later ones — the same temporal order it would face in
production. A random split would allow the model to learn from future inspections,
which is data leakage.

In [4]:
# Re-load with date to sort, then drop
df_full = pd.read_csv(BASE / 'feature_matrix.csv', parse_dates=['inspection_date'])
df_full = df_full.drop(columns=['ElevatingDevicesNumber', 'InspectionNumber', 'location'])
df_full[bool_cols] = df_full[bool_cols].astype(int)
df_full = df_full.sort_values('inspection_date').reset_index(drop=True)

# Derived ratio features — raw counts correlate with history length; ratios generalize
# better across time periods. clip(lower=1) avoids division by zero on first inspections.
df_full['pass_rate'] = (
    df_full['prior_outcome_counts_passed'] / df_full['prior_inspection_count'].clip(lower=1)
)
df_full['needs_action_rate'] = (
    df_full['prior_outcome_counts_needs_action'] / df_full['prior_inspection_count'].clip(lower=1)
)
df_full['orders_per_inspection'] = (
    df_full['prior_order_count'] / df_full['prior_inspection_count'].clip(lower=1)
)

split_idx = int(len(df_full) * 0.80)
train = df_full.iloc[:split_idx].drop(columns=['inspection_date'])
test  = df_full.iloc[split_idx:].drop(columns=['inspection_date'])

X_train = train.drop(columns=['outcome_binary'])
y_train = train['outcome_binary']
X_test  = test.drop(columns=['outcome_binary'])
y_test  = test['outcome_binary']

train_start = df_full['inspection_date'].iloc[0].date()
train_end   = df_full['inspection_date'].iloc[split_idx - 1].date()
test_start  = df_full['inspection_date'].iloc[split_idx].date()
test_end    = df_full['inspection_date'].iloc[-1].date()

print(f'Train: {len(X_train):,} rows  ({train_start} → {train_end})')
print(f'Test:  {len(X_test):,} rows  ({test_start} → {test_end})')
print(f'Features: {X_train.shape[1]}')
print(f'\nTest target distribution:')
print(y_test.value_counts())

Train: 113,431 rows  (2011-01-04 → 2015-12-15)
Test:  28,358 rows  (2015-12-15 → 2017-01-09)
Features: 33

Test target distribution:
outcome_binary
Needs Action    17852
Passed          10506
Name: count, dtype: int64


## 4. Model Evaluation — Without Feature Selection

In [5]:
# Each pipeline gets its own fresh clf via clone() — sharing a single estimator
# object across pipelines causes the with-selection fit to overwrite the
# no-selection pipeline's fitted state (feature count mismatch at predict time).
model_templates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    # Early stopping (n_iter_no_change + validation_fraction) prevents overfitting
    # to the 2011–2015 training distribution before applying to 2015–2017 test data.
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=2000, learning_rate=0.02, max_depth=5, min_samples_leaf=20,
        n_iter_no_change=30, validation_fraction=0.1, random_state=42
    ),
}

results = {}

for name, clf_template in model_templates.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', clone(clf_template)),
    ])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = {'no_selection': acc, 'pipe': pipe}
    print(f'{name}: {acc:.3f} ({acc:.1%})  — baseline: {baseline:.1%}  — beat by {acc - baseline:+.1%}')

LogisticRegression: 0.602 (60.2%)  — baseline: 62.0%  — beat by -1.8%
RandomForest: 0.580 (58.0%)  — baseline: 62.0%  — beat by -3.9%
HistGradientBoosting: 0.604 (60.4%)  — baseline: 62.0%  — beat by -1.6%


## 5. Model Evaluation — With Feature Selection

`SelectKBest` with `mutual_info_classif` ranks features by how much information each
shares with the target. `k=25` retains the 25 most informative features out of 33
(30 original + 3 derived ratio features), reducing noise and potentially improving
generalization.

In [6]:
K = 25

for name, clf_template in model_templates.items():
    pipe_fs = Pipeline([
        ('scaler', StandardScaler()),
        ('select', SelectKBest(mutual_info_classif, k=K)),
        ('clf', clone(clf_template)),
    ])
    pipe_fs.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe_fs.predict(X_test))
    results[name]['with_selection'] = acc
    results[name]['pipe_fs'] = pipe_fs
    print(f'{name} + SelectKBest(k={K}): {acc:.3f} ({acc:.1%})  — beat baseline by {acc - baseline:+.1%}')

LogisticRegression + SelectKBest(k=25): 0.602 (60.2%)  — beat baseline by -1.8%
RandomForest + SelectKBest(k=25): 0.579 (57.9%)  — beat baseline by -4.1%
HistGradientBoosting + SelectKBest(k=25): 0.603 (60.3%)  — beat baseline by -1.7%


## 6. Best Model and Feature Importance

In [7]:
print('=== Results Summary ===\n')
print(f'{"Model":<40} {"No selection":>14} {"With selection":>15} {"vs Baseline":>12}')
print('-' * 83)
for name, r in results.items():
    no_sel = r['no_selection']
    with_sel = r['with_selection']
    best = max(no_sel, with_sel)
    print(f'{name:<40} {no_sel:>13.1%} {with_sel:>14.1%} {best - baseline:>+11.1%}')
print(f'\nBaseline: {baseline:.1%}')

# Pick best overall
best_name = max(results, key=lambda n: max(results[n]['no_selection'], results[n]['with_selection']))
best_r = results[best_name]
best_acc = max(best_r['no_selection'], best_r['with_selection'])
# Ties go to the no-selection pipe (simpler model preferred)
best_pipe = best_r['pipe'] if best_r['no_selection'] >= best_r['with_selection'] else best_r['pipe_fs']

print(f'\nBest model: {best_name}')
print(f'Best accuracy: {best_acc:.3f} ({best_acc:.1%})')
print(f'Beats baseline by: {best_acc - baseline:+.1%}')

# Show selected features only when the with-selection pipe strictly outperformed
if best_r['with_selection'] > best_r['no_selection']:
    selector = best_pipe.named_steps['select']
    selected = X_train.columns[selector.get_support()].tolist()
    print(f'\nSelected features (k={K}):')
    for f in selected:
        print(f'  {f}')

=== Results Summary ===

Model                                      No selection  With selection  vs Baseline
-----------------------------------------------------------------------------------
LogisticRegression                               60.2%          60.2%       -1.8%
RandomForest                                     58.0%          57.9%       -3.9%
HistGradientBoosting                             60.4%          60.3%       -1.6%

Baseline: 62.0%

Best model: HistGradientBoosting
Best accuracy: 0.604 (60.4%)
Beats baseline by: -1.6%


## 7. Classification Report — Best Model

In [8]:
y_pred = best_pipe.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

Needs Action       0.66      0.77      0.71     17852
      Passed       0.45      0.33      0.38     10506

    accuracy                           0.60     28358
   macro avg       0.56      0.55      0.54     28358
weighted avg       0.58      0.60      0.59     28358



## 8. Threshold Calibration

At the default threshold (0.5), the best model predicts `Passed` whenever
`P(Passed) ≥ 0.5`. If the model's precision for `Passed` is below 50%, every
such prediction hurts overall accuracy — the model would do better by not
predicting `Passed` at all for those cases.

Raising the threshold means the model only commits to `Passed` when it is more
confident. This trades recall (fewer `Passed` predictions) for precision (higher
hit rate on the ones it does make). The sweet spot maximises accuracy.

**Note:** the sweep below uses the test set for illustration. In a production
setting the threshold would be tuned on a held-out validation set.

In [9]:
classes = list(best_pipe.classes_)
passed_idx = classes.index('Passed')
y_proba = best_pipe.predict_proba(X_test)

print(f'{"Threshold":>10}  {"Accuracy":>10}  {"vs Baseline":>12}  {"Passed predicted":>18}')
print('-' * 58)

best_thresh, best_acc_thresh = 0.5, 0.0
for thresh in np.arange(0.40, 0.85, 0.05):
    y_pred_t = np.where(y_proba[:, passed_idx] >= thresh, 'Passed', 'Needs Action')
    acc_t = accuracy_score(y_test, y_pred_t)
    n_passed = (y_pred_t == 'Passed').sum()
    beat = acc_t - baseline
    marker = '  ← beats baseline' if acc_t > baseline else ''
    if acc_t > best_acc_thresh:
        best_thresh, best_acc_thresh = thresh, acc_t
    print(f'{thresh:>10.2f}  {acc_t:>10.3f}  {beat:>+11.1%}  {n_passed:>18,}{marker}')

print(f'\nOptimal threshold: {best_thresh:.2f}')
print(f'Best accuracy:     {best_acc_thresh:.3f} ({best_acc_thresh:.1%})')
print(f'Beats baseline by: {best_acc_thresh - baseline:+.1%}')

y_pred_best = np.where(y_proba[:, passed_idx] >= best_thresh, 'Passed', 'Needs Action')
print()
print(classification_report(y_test, y_pred_best))

 Threshold    Accuracy   vs Baseline    Passed predicted
----------------------------------------------------------
      0.40       0.547        -7.3%              16,901
      0.45       0.578        -4.2%              12,222
      0.50       0.604        -1.6%               7,578
      0.55       0.620        +0.0%               3,208  ← beats baseline
      0.60       0.625        +0.5%               1,601  ← beats baseline
      0.65       0.627        +0.7%                 818  ← beats baseline
      0.70       0.628        +0.8%                 446  ← beats baseline
      0.75       0.629        +0.9%                 203  ← beats baseline
      0.80       0.629        +0.9%                  94  ← beats baseline

Optimal threshold: 0.75
Best accuracy:     0.629 (62.9%)
Beats baseline by: +0.9%

              precision    recall  f1-score   support

Needs Action       0.63      0.99      0.77     17852
      Passed       0.43      0.01      0.02     10506

    accuracy            